# 02 - Judge Personality

Runs two LLM judges over the raw Moltbook dataset: a persona judge and an assistant-safety
judge, across categories C (Socializing), A (Identity), D (Economics), F (Promotion),
E (Viewpoint), B (Technology) (`config.CADFEB_CATEGORIES`), targeting 200 posts/category for
each judge.

**Persona judge population**: toxicity in `[4, 3]`, priority-ordered
(`config.CADFEB_PERSONA_TOXICITY_LEVELS`) — each category's 200-post quota is filled by
exhausting toxicity==4 (Malicious) posts first, then backfilling with toxicity==3
(Manipulative) posts.

**Assistant judge population**: toxicity==0 only — plenty of data in every category to reach
the full 200/category.

**Carryover**: categories A/D/F/B were already sampled and judged at 50/category in an earlier
pass (results now under `archive/results/DBAF_results/`). This notebook loads that pass's
cached samples as `carryover` into `dataset.get_or_create_sample()`, keeping those posts and
only drawing the shortfall (up to 200/category) from the remaining pool, then copies the
already-judged rows forward via `ResultStore.copy_from()` so `run_judge()` skips them — only
genuinely new posts incur API calls. Categories C and E have no prior pass, so they're sampled
and judged fresh.

**Even combining toxicity 4+3, some categories can't reach 200** — a hard data ceiling, not a
sampling bug:

| category | toxicity=4 | toxicity=3 | combined |
|---|---|---|---|
| D | 254 | 201 | 455 (200 reachable on toxicity=4 alone) |
| B | 158 | 35  | 193 (falls short of 200 even combined) |
| A | 59  | 501 | 560 |
| F | 59  | 249 | 308 |
| E | 28  | 230 | 258 |
| C | 5   | 1672| 1677 |

Idempotent — safe to interrupt and re-run at any point. Output: `results/CADFEB_results/`.

In [1]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import pandas as pd

from src import config, dataset, prompts
from src.judge_schema import AgePersonaJudgeResult, AssistantJudgeResult
from src.categorizer import ResultStore, get_client, run_judge

## 1. Load posts (cached; downloads only if `data/raw/posts.parquet` is missing)

In [2]:
df = dataset.download_posts()
print(f"loaded {len(df)} rows")

client = get_client()

loaded 44376 rows


## 2. Persona judge (toxicity in [4, 3], priority-ordered)

### 2a. Sample: extend the prior DBAF persona sample (A/D/F/B) up to 200/category, add fresh samples for C/E.

In [3]:
dbaf_persona_sample_path = config.PROCESSED_DATA_DIR / f"{config.DBAF_PERSONA_SAMPLE_NAME}.parquet"
persona_carryover = pd.read_parquet(dbaf_persona_sample_path) if dbaf_persona_sample_path.exists() else None
if persona_carryover is not None:
    persona_carryover = persona_carryover[persona_carryover[config.COL_CATEGORY].isin(config.CADFEB_CATEGORIES)]
    print(f"carrying over {len(persona_carryover)} already-sampled persona posts from run 3 (DBAF)")
else:
    print("no run-3 DBAF persona sample found -- sampling all CADFEB categories fresh")

cadfeb_persona_sample = dataset.get_or_create_sample(
    df,
    name=config.CADFEB_PERSONA_SAMPLE_NAME,
    toxicity=config.CADFEB_PERSONA_TOXICITY_LEVELS,
    per_category=config.CADFEB_SAMPLES_PER_CATEGORY,
    categories=config.CADFEB_CATEGORIES,
    carryover=persona_carryover,
)
print(f"CADFEB persona sample: {len(cadfeb_persona_sample)} rows")
cadfeb_persona_sample.groupby([config.COL_CATEGORY, config.COL_TOXICITY]).size().unstack(fill_value=0)

carrying over 200 already-sampled persona posts from run 3 (DBAF)
CADFEB persona sample: 1193 rows


toxic_level,3,4
topic_label,,
A,141,59
B,35,158
C,195,5
D,0,200
E,172,28
F,141,59


### 2b. Copy already-judged rows forward, then judge the rest

In [4]:
cadfeb_persona_store = ResultStore(config.CADFEB_RESULTS_DIR / f"{config.CADFEB_PERSONA_JUDGE_NAME}.jsonl")

persona_allowed_ids = set(cadfeb_persona_sample[config.COL_POST_ID].astype(str))
n_copied = cadfeb_persona_store.copy_from(
    config.DBAF_RESULTS_DIR / f"{config.DBAF_PERSONA_JUDGE_NAME}.jsonl",
    allowed_ids=persona_allowed_ids,
)
print(f"copied {n_copied} already-judged persona rows from run 3")

run_judge(
    client=client,
    store=cadfeb_persona_store,
    rows=cadfeb_persona_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.AGE_PERSONA_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_age_persona_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AgePersonaJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"CADFEB persona judge: {len(cadfeb_persona_store.seen_ids())} successful rows cached")

copied 0 already-judged persona rows from run 3


judging (AgePersonaJudgeResult): 100%|██████████| 844/844 [40:44<00:00,  2.90s/it]  

CADFEB persona judge: 1193 successful rows cached


## 3. Assistant judge (toxicity==0)

### 3a. Sample: extend the prior DBAF assistant sample (A/D/F/B) up to 200/category, add fresh samples for C/E.

In [5]:
dbaf_assistant_sample_path = config.PROCESSED_DATA_DIR / f"{config.DBAF_ASSISTANT_SAMPLE_NAME}.parquet"
assistant_carryover = pd.read_parquet(dbaf_assistant_sample_path) if dbaf_assistant_sample_path.exists() else None
if assistant_carryover is not None:
    assistant_carryover = assistant_carryover[assistant_carryover[config.COL_CATEGORY].isin(config.CADFEB_CATEGORIES)]
    print(f"carrying over {len(assistant_carryover)} already-sampled assistant posts from run 3 (DBAF)")
else:
    print("no run-3 DBAF assistant sample found -- sampling all CADFEB categories fresh")

cadfeb_assistant_sample = dataset.get_or_create_sample(
    df,
    name=config.CADFEB_ASSISTANT_SAMPLE_NAME,
    toxicity=config.TOXICITY_SAFE,
    per_category=config.CADFEB_SAMPLES_PER_CATEGORY,
    categories=config.CADFEB_CATEGORIES,
    carryover=assistant_carryover,
)
print(f"CADFEB assistant sample: {len(cadfeb_assistant_sample)} rows")
cadfeb_assistant_sample[config.COL_CATEGORY].value_counts().sort_index()

carrying over 200 already-sampled assistant posts from run 3 (DBAF)
CADFEB assistant sample: 1200 rows


topic_label
A    200
B    200
C    200
D    200
E    200
F    200
Name: count, dtype: int64

### 3b. Copy already-judged rows forward, then judge the rest

In [6]:
cadfeb_assistant_store = ResultStore(config.CADFEB_RESULTS_DIR / f"{config.CADFEB_ASSISTANT_JUDGE_NAME}.jsonl")

assistant_allowed_ids = set(cadfeb_assistant_sample[config.COL_POST_ID].astype(str))
n_copied = cadfeb_assistant_store.copy_from(
    config.DBAF_RESULTS_DIR / f"{config.DBAF_ASSISTANT_JUDGE_NAME}.jsonl",
    allowed_ids=assistant_allowed_ids,
)
print(f"copied {n_copied} already-judged assistant rows from run 3")

run_judge(
    client=client,
    store=cadfeb_assistant_store,
    rows=cadfeb_assistant_sample.to_dict("records"),
    id_key=config.COL_POST_ID,
    system_prompt=prompts.ASSISTANT_JUDGE_SYSTEM,
    build_user_prompt=lambda row: prompts.build_assistant_prompt(
        post_id=str(row[config.COL_POST_ID]), content=row[config.COL_CONTENT]
    ),
    result_model=AssistantJudgeResult,
    content_key=config.COL_CONTENT,
    category_key=config.COL_CATEGORY,
)
print(f"CADFEB assistant judge: {len(cadfeb_assistant_store.seen_ids())} successful rows cached")

copied 187 already-judged assistant rows from run 3


judging (AssistantJudgeResult): 100%|██████████| 1013/1013 [52:52<00:00,  3.13s/it] 

CADFEB assistant judge: 1140 successful rows cached


## 4. Error sanity check

Error rows are retried automatically on the next run of this notebook.

In [7]:
for name, store in [("CADFEB persona", cadfeb_persona_store), ("CADFEB assistant", cadfeb_assistant_store)]:
    rows = store.load_all_rows()
    errors = [r for r in rows if "error" in r]
    print(f"{name}: {len(rows)} total rows, {len(errors)} errors, {len(rows) - len(errors)} successful")
    for e in errors[:5]:
        print("  ", e)

CADFEB persona: 1193 total rows, 0 errors, 1193 successful
CADFEB assistant: 1200 total rows, 60 errors, 1140 successful
   {'post_id': '597bdf3a-dace-44fe-a124-3a36ae681c0b', 'error': "validation_error: 3 validation errors for AssistantJudgeResult\nis_safe_for_assistant\n  Field required [type=missing, input_value={'description': 'Output o...44fe-a124-3a36ae681c0b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nconfidence\n  Field required [type=missing, input_value={'description': 'Output o...44fe-a124-3a36ae681c0b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing\nrationale\n  Field required [type=missing, input_value={'description': 'Output o...44fe-a124-3a36ae681c0b'}, input_type=dict]\n    For further information visit https://errors.pydantic.dev/2.13/v/missing"}
   {'post_id': '5f32d58a-fc00-409f-89fa-4bf2ec90d4b7', 'error': "validation_error: 3 validation errors for AssistantJudge

## 5. Summary: personas + safety per category

Uses the `category` field stored directly in each result row -- no join needed.

In [8]:
cadfeb_persona_rows = [r for r in cadfeb_persona_store.load_all_rows() if "error" not in r]
cadfeb_persona_df = pd.DataFrame(cadfeb_persona_rows)

persona_by_category = (
    cadfeb_persona_df.groupby(["category", "persona"]).size().unstack(fill_value=0)
)
print("Persona counts per category:")
persona_by_category

Persona counts per category:


persona,evil,malicious-manipulative,other,sycophantic
category,,,,
A,101,56,38,5
B,6,134,50,3
C,6,13,149,32
D,4,183,12,1
E,118,31,47,4
F,26,110,60,4


In [9]:
cadfeb_assistant_rows = [r for r in cadfeb_assistant_store.load_all_rows() if "error" not in r]
cadfeb_assistant_df = pd.DataFrame(cadfeb_assistant_rows)

safety_by_category = (
    cadfeb_assistant_df.groupby("category")["is_safe_for_assistant"]
    .agg(["sum", "count"])
    .rename(columns={"sum": "safe_count", "count": "total"})
)
safety_by_category["safe_rate"] = safety_by_category["safe_count"] / safety_by_category["total"]
print("Assistant-safety rate per category:")
safety_by_category

Assistant-safety rate per category:


,safe_count,total,safe_rate
category,,,
A,173,197,0.878173
B,180,188,0.957447
C,178,188,0.946809
D,96,189,0.507937
E,183,193,0.948187
F,152,185,0.821622
